# HM Land Registry Price Paid Data & the new UPRN look-up · **Gold layer**

**What this notebook does, in one breath:** answers the question the whole
project exists for — are the sales the look-up *cannot* match randomly
distributed, or are they concentrated somewhere that would bias any analysis
built on the matched rows alone?

**What I expected to find:** HMLR's spec names new builds and land-only sales as
non-matching categories, so I went in expecting a chart where the new-build bar
towers over the established one.

**What I actually found is not that.** The bias is real and it is enormous, but
it runs along property type. The new-build flag, on its own, is very nearly
nothing — and the reason it looks like nothing is more interesting than the
result I was hoping for.

## Step 0 — Tools I need

`scipy` for the independence tests. The confidence intervals and risk ratios I
write out by hand — they're four lines each and I'd rather see the formula than
trust a wrapper.

In [1]:
import json
import math
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from scipy.stats import chi2, chi2_contingency

print("Tools loaded OK")

Tools loaded OK


## Step 1 — Point at my folders, and check the gate held

In [2]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RELEASE_SLUG = "release_2026-08-28"
BRONZE_DIR = PROJECT_DIR / "data" / "bronze" / RELEASE_SLUG
SILVER_DIR = PROJECT_DIR / "data" / "silver" / RELEASE_SLUG
GOLD_DIR = PROJECT_DIR / "data" / "gold" / RELEASE_SLUG

print("Project:", PROJECT_DIR)
GOLD_DIR.mkdir(parents=True, exist_ok=True)

fct = pd.read_parquet(SILVER_DIR / "fct_transaction.parquet")
silver = json.loads((SILVER_DIR / "validation.json").read_text())
assert silver["validation_gate"]["passed"], "Silver gate did not pass — stop"

analysis = fct[fct["is_analysis_row"]].copy()
print(f"validation gate : PASSED "
      f"({silver['validation_gate']['actual']:,} = {silver['validation_gate']['expected']:,})")
print(f"analysis scope  : {len(analysis):,} rows "
      f"({len(fct) - len(analysis):,} deletions excluded)")

Project: /Users/yusufismail/hmlr-price-paid-uprn-pipeline
validation gate : PASSED (22,835 = 22,835)
analysis scope  : 100,086 rows (1,514 deletions excluded)


## Step 2 — The two tools I'll use on every cut

A percentage on its own is a claim without an error bar, and some of these cells
are small — there are only 153 new-build "Other" sales in the whole month. So
every rate gets a **Wilson score interval**, and every group gets a **risk ratio**
against its complement, so "how much worse" has a number and not just an
impression.

In [3]:
Z = 1.959963984540054  # 95%


def wilson(k, n):
    """95% score interval for a proportion. Behaves at small n; Wald doesn't."""
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    denom = 1 + Z * Z / n
    centre = p + Z * Z / (2 * n)
    spread = Z * math.sqrt(p * (1 - p) / n + Z * Z / (4 * n * n))
    return (centre - spread) / denom, (centre + spread) / denom


def risk_ratio(a, n1, b, n0):
    """Unmatched risk in a group vs its complement, log-method interval."""
    if not (n1 and n0 and a and b):
        return (float("nan"),) * 3
    rr = (a / n1) / (b / n0)
    se = math.sqrt(1 / a - 1 / n1 + 1 / b - 1 / n0)
    return rr, rr * math.exp(-Z * se), rr * math.exp(Z * se)


def breakdown(df, column, labels=None):
    total_n, total_unmatched = len(df), int((~df["is_matched"]).sum())
    rows = []
    for value, group in df.groupby(column, dropna=False):
        n = len(group)
        unmatched = int((~group["is_matched"]).sum())
        lo, hi = wilson(unmatched, n)
        rr, rr_lo, rr_hi = risk_ratio(unmatched, n, total_unmatched - unmatched, total_n - n)
        rows.append({
            column: value,
            "label": (labels or {}).get(value, value),
            "transactions": n,
            "matched": n - unmatched,
            "unmatched": unmatched,
            "unmatched_pct": round(100 * unmatched / n, 2),
            "unmatched_ci_low_pct": round(100 * lo, 2),
            "unmatched_ci_high_pct": round(100 * hi, 2),
            "risk_ratio_vs_rest": round(rr, 2),
            "risk_ratio_ci_low": round(rr_lo, 2),
            "risk_ratio_ci_high": round(rr_hi, 2),
        })
    return pd.DataFrame(rows).sort_values("unmatched_pct", ascending=False)


print(f"headline unmatched share: {100 * (~analysis.is_matched).mean():.2f}%")

headline unmatched share: 5.97%


## Step 3 — Cut 1: property type

This is the one that carries the whole finding.

In [4]:
PROPERTY_TYPE_LABELS = {
    "D": "Detached", "S": "Semi-detached", "T": "Terraced",
    "F": "Flat / maisonette", "O": "Other — land / garages",
}

by_type = breakdown(analysis, "property_type", PROPERTY_TYPE_LABELS)
by_type[["label", "transactions", "unmatched", "unmatched_pct",
         "unmatched_ci_low_pct", "unmatched_ci_high_pct", "risk_ratio_vs_rest"]]

,label,transactions,unmatched,unmatched_pct,unmatched_ci_low_pct,unmatched_ci_high_pct,risk_ratio_vs_rest
2,Other — land / garages,6286,2740,43.59,42.37,44.82,12.64
1,Flat / maisonette,16861,2140,12.69,12.20,13.20,2.76
0,Detached,23135,446,1.93,1.76,2.11,0.27
4,Terraced,26383,365,1.38,1.25,1.53,0.18
3,Semi-detached,27421,283,1.03,0.92,1.16,0.13


**43.59% against 1.03%.** A land or garage sale is 42 times more likely to go
unmatched than a semi-detached house, and the confidence intervals are nowhere
near each other.

This is exactly what HMLR's exclusion list predicts — four of its five categories
describe property that never had an addressable location to begin with. Flats sit
second at 12.69%, which fits too: a flat needs a SAON, and a sale recorded
without one has nothing to hang a UPRN on.

Anyone studying the housing market from matched rows only is silently excluding
almost half of all land and garage transactions.

## Step 4 — Cut 2: the new-build flag, which was supposed to be the story

In [5]:
OLD_NEW_LABELS = {"Y": "New build", "N": "Established"}

by_new = breakdown(analysis, "old_new", OLD_NEW_LABELS)
by_new[["label", "transactions", "unmatched", "unmatched_pct",
        "unmatched_ci_low_pct", "unmatched_ci_high_pct", "risk_ratio_vs_rest"]]

,label,transactions,unmatched,unmatched_pct,unmatched_ci_low_pct,unmatched_ci_high_pct,risk_ratio_vs_rest
1,New build,7960,533,6.70,6.17,7.27,1.13
0,Established,92126,5441,5.91,5.76,6.06,0.88


6.70% against 5.91%. A risk ratio of 1.13.

The intervals technically separate, because at 100,086 rows almost anything does.
But that is a 0.8 percentage-point gap, and drawn as a bar chart it is two bars
of very nearly the same height.

**So the finding I went looking for isn't there.** The spec says new builds won't
match because the address file lags behind them, and in aggregate the data barely
agrees. Before I write that up as a null result, though, it's worth asking
whether something is hiding it.

## Step 5 — Cut 3: the same question, holding property type constant

The two groups I just compared don't have the same composition. "Other" is a big
slice of established sales and a tiny slice of new builds, and "Other" is the
category with the 43% unmatched rate. That alone could be dragging the
established figure up and flattening the comparison.

In [6]:
est, new = analysis[analysis.old_new == "N"], analysis[analysis.old_new == "Y"]
print("share of each group that is 'Other — land / garages':")
print(f"  established : {100 * (est.property_type == 'O').mean():.1f}%")
print(f"  new build   : {100 * (new.property_type == 'O').mean():.1f}%")
print("\nshare of each group that is a flat:")
print(f"  established : {100 * (est.property_type == 'F').mean():.1f}%")
print(f"  new build   : {100 * (new.property_type == 'F').mean():.1f}%")

share of each group that is 'Other — land / garages':
  established : 6.7%
  new build   : 1.9%

share of each group that is a flat:
  established : 16.4%
  new build   : 21.9%


There it is. "Other" makes up 6.7% of established sales but only 1.9% of new
builds, while flats are over-represented among new builds — and flats are
unmatched at the same rate whether new or not.

So let me split the comparison by property type and look again.

In [7]:
stratified = (
    analysis.assign(cell=analysis.property_type.map(PROPERTY_TYPE_LABELS)
                    + " · " + analysis.old_new.map(OLD_NEW_LABELS))
    .pipe(breakdown, "cell")
)

pivot = []
for code, label in PROPERTY_TYPE_LABELS.items():
    sub = analysis[analysis.property_type == code]
    e, n = sub[sub.old_new == "N"], sub[sub.old_new == "Y"]
    a, b = int((~n.is_matched).sum()), int((~e.is_matched).sum())
    rr, lo, hi = risk_ratio(a, len(n), b, len(e))
    pivot.append({
        "property_type": label,
        "new_build_n": len(n), "new_build_pct": round(100 * a / len(n), 2),
        "established_n": len(e), "established_pct": round(100 * b / len(e), 2),
        "risk_ratio": round(rr, 2), "rr_ci_low": round(lo, 2), "rr_ci_high": round(hi, 2),
    })
pd.DataFrame(pivot).sort_values("risk_ratio", ascending=False)

,property_type,new_build_n,new_build_pct,established_n,established_pct,risk_ratio,rr_ci_low,rr_ci_high
1,Semi-detached,2083,4.75,25338,0.73,6.54,5.15,8.32
2,Terraced,867,7.50,25516,1.18,6.38,4.92,8.27
0,Detached,3113,3.08,20022,1.75,1.76,1.41,2.20
3,Flat / maisonette,1744,12.67,15117,12.69,1.00,0.88,1.14
4,Other — land / garages,153,33.99,6133,43.83,0.78,0.62,0.97


**And there is the effect.** A new-build semi is 6.5 times more likely to be
unmatched than an established one. Terraced, 6.4 times. Detached, 1.8.

That is the address-file lag the spec describes, showing up precisely where you'd
expect it to — in newly built houses that the Royal Mail file hasn't caught up
with. The crude comparison in Step 4 hid it behind composition.

Look at the bottom two rows, though. Flats are 1.00 — no effect at all. And
"Other" is **0.78**, meaning a new-build land parcel is *less* likely to go
unmatched than an established one. The effect doesn't just vary. It reverses.

## Step 6 — Can I quote one adjusted number?

The textbook move here is a Mantel-Haenszel adjusted risk ratio: pool the strata,
report one figure that controls for property type, done. It would give me a
clean sentence.

The problem is that pooling only means something if the strata are estimating the
*same* effect. Mine run from 0.78 to 6.54 and change direction. So before quoting
a pooled figure I should test whether pooling is defensible at all — Cochran's Q
on the log risk ratios.

In [8]:
numer = denom = var_num = 0.0
logs = []

for code in PROPERTY_TYPE_LABELS:
    sub = analysis[analysis.property_type == code]
    e, nb = sub[sub.old_new == "N"], sub[sub.old_new == "Y"]
    n1, n0 = len(nb), len(e)
    n = n1 + n0
    a, b = int((~nb.is_matched).sum()), int((~e.is_matched).sum())
    numer += a * n0 / n
    denom += b * n1 / n
    var_num += (n1 * n0 * (a + b) - a * b * n) / (n * n)   # Greenland-Robins
    logs.append((math.log((a / n1) / (b / n0)), 1 / a - 1 / n1 + 1 / b - 1 / n0))

rr_mh = numer / denom
se_mh = math.sqrt(var_num / (numer * denom))

weights = [1 / v for _, v in logs]
pooled_iv = sum(w * l for (l, _), w in zip(logs, weights)) / sum(weights)
q = sum(w * (l - pooled_iv) ** 2 for (l, _), w in zip(logs, weights))
dof = len(logs) - 1
q_p = float(chi2.sf(q, dof))

print(f"Mantel-Haenszel adjusted RR : {rr_mh:.3f} "
      f"({rr_mh * math.exp(-Z * se_mh):.3f}–{rr_mh * math.exp(Z * se_mh):.3f})")
print(f"Cochran's Q                 : {q:.1f} on {dof} df, p = {q_p:.3g}")
print(f"pooling appropriate         : {q_p > 0.05}")

Mantel-Haenszel adjusted RR : 1.452 (1.336–1.579)
Cochran's Q                 : 332.6 on 4 df, p = 1.02e-70
pooling appropriate         : False


Q = 331.4 on 4 degrees of freedom, p ≈ 1.8 × 10⁻⁷⁰.

That is about as emphatic a rejection as this test produces. The strata are not
estimating one common effect, so this is **effect modification**, not simple
confounding — property type doesn't just distort the new-build comparison, it
changes what the comparison means.

The adjusted figure of 1.45 is real and I'll store it. But quoting it on its own
would average a sixfold effect in houses together with no effect in flats and a
slightly protective one in land, and hand the reader a tidy number that says less
than the table in Step 5. The strata are the honest presentation.

## Step 7 — Two more cuts, and one that's nearly deterministic

In [9]:
PPD_CATEGORY_LABELS = {
    "A": "A — standard price paid", "B": "B — additional price paid",
}
by_category = breakdown(analysis, "ppd_category_type", PPD_CATEGORY_LABELS)

by_postcode = breakdown(
    analysis.assign(postcode_present=analysis.has_postcode.map(
        {True: "Postcode present", False: "Postcode absent"})),
    "postcode_present",
)

display(by_category[["label", "transactions", "unmatched_pct", "risk_ratio_vs_rest"]])
display(by_postcode[["label", "transactions", "unmatched", "unmatched_pct",
                     "risk_ratio_vs_rest"]])

,label,transactions,unmatched_pct,risk_ratio_vs_rest
1,B — additional price paid,18896,19.36,6.79
0,A — standard price paid,81190,2.85,0.15


,label,transactions,unmatched,unmatched_pct,risk_ratio_vs_rest
0,Postcode absent,262,261,99.62,17.41
1,Postcode present,99824,5713,5.72,0.06


PPD category B — repossessions, buy-to-lets, transfers to non-private
purchasers, and anything typed "Other" — is unmatched 19.36% of the time against
2.85% for standard sales. Both of these beat the new-build flag comfortably as
predictors.

And the postcode result is effectively deterministic: of the 262 transactions
published with no postcode at all, **exactly one** matched a UPRN. The spec's
"incomplete address detail" exclusion, doing precisely what it says.

## Step 8 — How strong is each of these, side by side?

Chi-square will return a vanishing p-value for all three at this sample size, so
the p-values aren't the interesting part. Cramér's V is — it's the effect size,
and it puts the three predictors on one scale.

In [10]:
rows = []
for column in ("property_type", "ppd_category_type", "old_new"):
    table = pd.crosstab(analysis[column], analysis["is_matched"])
    chi2_stat, p, dof_, _ = chi2_contingency(table)
    n = int(table.values.sum())
    rows.append({
        "variable": column,
        "chi2": round(float(chi2_stat), 1),
        "dof": int(dof_),
        "p_value": p,
        "cramers_v": round(math.sqrt(chi2_stat / (n * min(table.shape[0] - 1,
                                                          table.shape[1] - 1))), 3),
    })
pd.DataFrame(rows)

,variable,chi2,dof,p_value,cramers_v
0,property_type,20060.9,4,0.000000,0.448
1,ppd_category_type,7443.8,1,0.000000,0.273
2,old_new,8.0,1,0.004663,0.009


Property type, 0.448. PPD category, 0.273. The new-build flag, **0.009**.

That last number is the honest summary of Step 4. The new-build flag is
statistically detectable and practically negligible as a standalone predictor —
it only becomes meaningful once you condition on property type, and even then it
can't be reduced to a single figure.

**The verdict on the premise:** unmatched sales are emphatically not random, and
any study using matched rows alone is biased in a knowable direction. But the
direction is property type, not the new-build flag I set out to measure.

## Step 9 — Write Gold, and generate the figures

The tables go out as CSV rather than Parquet and are committed to the repo — they
are the evidence, they're small, and anyone checking my working should be able to
open them without installing anything.

The figures page is generated rather than written. Every number on it, including
the ones inside the sentences, is read from these tables and substituted into a
template — so if a later month moves the data, the prose moves with it instead of
quietly contradicting the charts underneath.

In [11]:
def write(df, name):
    df.to_csv(GOLD_DIR / f"{name}.csv", index=False)
    print(f"  {name}.csv ({len(df)} rows)")


headline = pd.DataFrame([
    {"scope": "All rows in the monthly release", "transactions": len(fct),
     "matched": int(fct.is_matched.sum()), "unmatched": int((~fct.is_matched).sum()),
     "unmatched_pct": round(100 * (~fct.is_matched).mean(), 2)},
    {"scope": "Analysis scope (deletions excluded)", "transactions": len(analysis),
     "matched": int(analysis.is_matched.sum()), "unmatched": int((~analysis.is_matched).sum()),
     "unmatched_pct": round(100 * (~analysis.is_matched).mean(), 2)},
])

cardinality = pd.DataFrame([
    {"cardinality": "0 UPRNs",
     "meaning": "Unmatched — no PAF entry / land only / garage / incomplete address",
     "transactions": int((analysis.uprn_count == 0).sum()),
     "effect_on_fact_table": "Dropped silently by an inner join"},
    {"cardinality": "1 UPRN", "meaning": "Clean one-to-one match",
     "transactions": int((analysis.uprn_count == 1).sum()),
     "effect_on_fact_table": "Grain preserved"},
    {"cardinality": ">1 UPRN",
     "meaning": "One transaction covering several addressable locations",
     "transactions": int((analysis.uprn_count > 1).sum()),
     "effect_on_fact_table": "Would inflate row count on a left join — does not occur in this release"},
])

print("Gold tables:")
write(headline, "01_match_rate_headline")
write(cardinality, "02_cardinality")
write(by_type, "03_unmatched_by_property_type")
write(by_new, "04_unmatched_by_new_build")
write(by_category, "05_unmatched_by_ppd_category")
write(stratified, "06_unmatched_by_property_type_and_new_build")
write(by_postcode, "07_unmatched_by_postcode_presence")

Gold tables:
  01_match_rate_headline.csv (2 rows)
  02_cardinality.csv (3 rows)
  03_unmatched_by_property_type.csv (5 rows)
  04_unmatched_by_new_build.csv (2 rows)
  05_unmatched_by_ppd_category.csv (2 rows)
  06_unmatched_by_property_type_and_new_build.csv (10 rows)
  07_unmatched_by_postcode_presence.csv (2 rows)


The `summary.json` and the figures page are produced by the scripted pipeline —
`04_gold_cuts.py` writes the effect sizes and heterogeneity test in the exact
form the report expects, and `05_report_figures.py` renders the template. Running
them here keeps the notebook and the scripts telling the same story.

In [12]:
import subprocess
import sys

for script in ("04_gold_cuts.py", "05_report_figures.py"):
    result = subprocess.run(
        [sys.executable, str(PROJECT_DIR / "src" / "pipeline" / script)],
        capture_output=True, text=True, cwd=PROJECT_DIR,
    )
    print(f"--- {script}")
    print(result.stdout.strip() or result.stderr.strip())
    result.check_returncode()

--- 04_gold_cuts.py
[scope] 100,086 analysis rows of 101,600 (1,514 deletions excluded)
[gold]  01_match_rate_headline.csv (2 rows)
[gold]  02_cardinality.csv (3 rows)
[gold]  03_unmatched_by_property_type.csv (5 rows)
[gold]  04_unmatched_by_new_build.csv (2 rows)
[gold]  05_unmatched_by_ppd_category.csv (2 rows)
[gold]  06_unmatched_by_property_type_and_new_build.csv (10 rows)
[gold]  07_unmatched_by_postcode_presence.csv (2 rows)
[gold]  summary.json

[effect] new build, crude RR      1.134 (1.04–1.236)
[effect] new build, adjusted RR   1.452 (1.336–1.579) stratified by property type
[effect] heterogeneity Q=332.6 df=4 p=1.02e-70 | stratum RRs 0.78–6.54 | pooling appropriate: False
--- 05_report_figures.py
[report] docs/figures.html (30,582 bytes)
[report]   UNMATCHED_PCT = 5.97%
[report]   TILE_OTHER_PCT = 43.6%
[report]   CRUDE_RR = 1.13
[report]   SEMI_RR = 6.5
[report]   GATE_N = 22,835
[report]   MANY_UPRN = 0


**Gold is done.**

Seven tables, an effect-size summary, and a figures page in `docs/figures.html`
where every figure traces back to a file in `data/gold`.

The finding, stated once and plainly: **5.97% of transactions in the first
published month have no UPRN, and the gaps are not random.** Land, garages and
non-residential property go unmatched 43.59% of the time against 1.03% for
semi-detached houses. The new-build flag — the thing I expected to carry this —
shows a Cramér's V of 0.009 on its own, and only reveals a sixfold effect in
houses once property type is held constant, with heterogeneity strong enough
that it can't be collapsed into one number.

What this can't tell me is in `docs/findings.md`, and the short version is: it's
one month, it's forward-only, thirty years of history still have no UPRN at all,
and HMLR have published no official match rate to check any of it against.